In [1]:
#| hide
from slopometer.score import *


# slopometer

> Score prose for simplicity and precision against plain-English reference rules

`slopometer` measures prose against the aai reference-prose rules defined in `aai_coding.write_docs`. The rules merge GOV.UK/GDS house style with the discipline of ASD-STE100. It covers 26 numbered tells, banned-word lists, and the reference register itself. The meter is for anyone shipping READMEs, docstrings, PR text, or agent responses. It catches slop mechanically, in milliseconds, with no LLM in the loop. Every finding cites a rule, quotes a span, and carries a weight. A document reduces to two numbers: density, the weighted findings per 100 prose words, and the single worst finding.


## Install

    pip install slopometer

The first score on a machine downloads spaCy's `en_core_web_md` model (about 40MB, once) into `~/.cache/slopometer` and loads it by path. The model never enters a virtual environment. One copy serves every project. Environment syncs cannot remove it.

## Use

In [2]:
from slopometer.score import score_text, score_path

In [3]:
score_text("This section describes our approach. It isn't just a linter - it's a comprehensive paradigm for quality.")

density 252.9 (weight 43 on 17 prose words), worst 10
1: [10] notxbuty (tell 16, not-X-but-Y): "isn't just a"
1: [10] splice (tell 1, splices): ' - '
1: [10] banned: 'comprehensive' -> 'complete'
1: [10] banned: 'paradigm'
1: [3] throat_clearing (tell 13, throat-clearing): 'This section describes'

`score_path` does the same for a file, and its rows carry `lineno|hash|` addresses in the exhash format, ready for hash-verified editors. The command line wraps both:

    slopometer README.md
    git log -1 --format=%B | slopometer
    slopometer draft.md --threshold 10

The threshold turns the density into an exit code for CI. The command runs warm through `warmpy`. The first call loads the model in a background process. Later calls answer in milliseconds. After thirty idle minutes the process exits.

## What it checks, and what it cannot

The rules live in notebooks that teach each family beside its code. Each rule states its tell, shows a violating example, and shows the plain rewrite. The [lexicon notebook](https://AnswerDotAI.github.io/slopometer/lexicon.html) holds the word and phrase rules. The [syntax notebook](https://AnswerDotAI.github.io/slopometer/syntax.html) builds sentence rules on spaCy's parse. The [para notebook](https://AnswerDotAI.github.io/slopometer/para.html) builds its rules on word vectors. The [score notebook](https://AnswerDotAI.github.io/slopometer/score.html) assembles the pipeline. A drift test asserts that every `write_docs` tell maps to a rule or to an explicit unscoreable registry. The meter and the style guide cannot drift apart silently.

A rule ships only when its false-positive rate on clean reference prose is near zero. The meter scores the style guide's own clean passage at exactly 0.0. The score notebook measures the blind spot instead of hiding it: mechanically chopped prose passes every surface rule while staying opaque. Agent review (`check_docs`) and an optional, explicitly invoked [Pangram](https://www.pangram.com) check cover that residue. Vale, write-good, and proselint solve neighboring problems. The [lexicon notebook](https://AnswerDotAI.github.io/slopometer/lexicon.html) records what came from them. It also credits the [GOV.UK words-to-avoid list](https://www.gov.uk/guidance/style-guide/a-to-z-of-gov-uk-style#words-to-avoid) (OGL v3).
